In [ ]:
import os, sys, time, subprocess
def W(msg):
    with open('/kaggle/working/trace.log', 'a') as f: f.write(f'[{time.strftime("%H:%M:%S")}] {msg}\n')
    print(msg, flush=True)
W('=== nb166 START ===')
import torch
cc = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0,0)
W(f'torch={torch.__version__} cc={cc} device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')

In [ ]:
# Force-reinstall torch 2.4.0+cu121 for P100 compatibility
W('=== INSTALL torch 2.4.0 ===')
t0 = time.time()
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall',
                    'torch==2.4.0', 'torchvision==0.19.0',
                    '--index-url', 'https://download.pytorch.org/whl/cu121'],
                   capture_output=True, text=True, timeout=900)
W(f'torch rc={r.returncode} elapsed={time.time()-t0:.0f}s')
if r.returncode != 0: W(f'stderr: {r.stderr[-1500:]}')

t1 = time.time()
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'boltz'],
                   capture_output=True, text=True, timeout=900)
W(f'boltz rc={r.returncode} elapsed={time.time()-t1:.0f}s')
if r.returncode != 0: W(f'stderr: {r.stderr[-1500:]}')
W('=== INSTALL END ===')

In [ ]:
# Validate via SUBPROCESS (separate Python process picks up new torch)
W('=== TORCH SUBPROCESS CHECK ===')
r = subprocess.run([sys.executable, '-c',
    'import torch; print(f"torch={torch.__version__}"); print(f"cuda={torch.cuda.is_available()}"); '
    'x = torch.zeros(1, device="cuda"); print(f"alloc OK: {x}"); '
    'print(f"cc={torch.cuda.get_device_capability(0)}")'],
    capture_output=True, text=True, timeout=60)
W(f'rc={r.returncode}')
W(f'stdout: {r.stdout}')
if r.returncode != 0: W(f'stderr: {r.stderr[-1500:]}')
W('=== TORCH CHECK END ===')

In [ ]:
from pathlib import Path
PXR_SEQ = 'LDRRTVVPATQHVTGTAYIWYRSGLCEHHIVEAATRGNVMTPSCKLITEELLGRPVHIVQPVKAVCSIVKQSDCRPFNQRSFKKYFTMENKVMVLNQELIKLALNFKLQDGRPHGGIIYDLSGEEDPKSWIWEVLEAWDIKAQVGPVTYAVTSLPFLQLSQYLDQDLALYIHQAFRYGPNALLDLLTDTRKHADRLELNGLAIRLLPELEVALMLLTQHTLREEKAGNFETIAEPFNALVMQVMEGYREKDPEAKQNQELHIWANKTKDPLLLEAHALDQFSCK'
yaml_str = f'version: 1\nsequences:\n- protein:\n    id: A\n    sequence: {PXR_SEQ}\n- ligand:\n    id: B\n    smiles: CCOc1ccccc1\nproperties:\n- affinity:\n    binder: B\n'
Path('/kaggle/working/test.yaml').write_text(yaml_str)
OUT = Path('/kaggle/working/o_test'); OUT.mkdir(exist_ok=True)
cmd = ['boltz', 'predict', '/kaggle/working/test.yaml', '--out_dir', str(OUT),
       '--use_msa_server', '--diffusion_samples', '1', '--recycling_steps', '1', '--sampling_steps', '50']
W(f'=== PREDICT: {" ".join(cmd)}')
t0 = time.time()
r = subprocess.run(cmd, capture_output=True, text=True, timeout=3600)
W(f'predict rc={r.returncode} elapsed={(time.time()-t0)/60:.1f}min')
W(f'stdout tail 2000: {r.stdout[-2000:]}')
if r.returncode != 0: W(f'stderr tail 2000: {r.stderr[-2000:]}')
import json
for jf in OUT.rglob('*affinity*.json'):
    W(f'AFF {jf.name}: {open(jf).read()[:500]}')
W('=== nb166 END ===')